In [ ]:
''' This notebook will load a pyTorch FNO model, as well as a dataset from ROOT and 
then plot percentile distributions of the energy density in the transverse plane
at specifical tau steps. It will also show percentile evelopes of the distributions.

Note that this notebook assumes that the model is trained on data scaed by tau.
Also note that the model definition must match that used to train the weights 
loaded from the model_input file.
'''

# libraries used
import os
import numpy as np
from os import path
import uproot
import awkward as ak
import torch
import sys
sys.path.append('../loc_libs')
from contour_v_plot import *

model_input    = '/home/davidstewart/FNO_JS3/junepub/train_models/etau/4types_500each/best_model_state_dict.pt'
input_file = '/home/davidstewart/JETSCAPE/config/bulk_writer/sample100_0_10_nw9p6_flat_xy60_t60.root'

# input file paraemters
ngrid = 60 # grid size in x and y
nT = 60 # number of time steps
odir = 'model_inspect' # will put log file output and generated plots here
tree_name = 't'
branch_name = 'user_res'


nevents = 30 # how many events to load
last_time_step = None # if None, will use the last time step in the data

# Train the model on input data from data_input/ directory
# save data to a new output directory (which will hold the saved model parameters)

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

if not os.path.exists(odir):
    os.makedirs(odir)
log = open(f'{odir}/log.txt','w')
log.write(f'model input:     {model_input} \n')
log.write(f'Input raw file:  {input_file}\n')
log.close()


In [ ]:
branch = uproot.open(input_file)[tree_name][branch_name]
if nevents > branch.num_entries:
    nevents = branch.num_entries
arr = np.reshape(ak.to_numpy(branch.array(library='ak', entry_stop=nevents)), (nevents, 3, ngrid, ngrid, nT))#, nevents=1000))
# arr = np.reshape(ak.to_numpy(branch.array(library='ak', entry_stop=nevents)), (nevents, 1, ngrid, ngrid, nT))#, nevents=1000))

if last_time_step:
    arr = arr[:,:,:,:, :last_time_step]

print(type(arr))
print(arr.shape)

# scale the input array by tau:
tau0 = 0.5
taustep = 0.1
taus_scaling = tau0 + np.arange(arr.shape[-1]) * taustep
arr[:,0,:,:,:] *= taus_scaling

# break into a training and verification set
data_etau_truth = arr

In [ ]:
# get the input model
from neuralop.models import FNO
model = FNO(
    in_channels=3,        # Input channels (e.g., velocity field)
    out_channels=3,       # Output channels
    # positional_embedding=GridEmbeddingND(in_channels=3, dim=3, grid_boundaries=[[-15,15],[-15,15],[3.5,3.5001]]),
    #  positional_embedding=GridEmbeddingND(in_channels=3, dim=3, grid_boundaries=[[-15,15],[-15,15],[0.6,15.5]]),
    n_modes=[30,30,25],   # [60,60,50],    # Number of modes in each layer
    hidden_channels=64,   # 20             # Width of the network
    projection_channel_ratio=2
).to(device)

model.load_state_dict(torch.load(model_input, map_location=device))
model.eval()

In [ ]:
# use the model to get predictions on the input data
def get_xy(in_data, i):
    time_steps=in_data.shape[-1]
    x = in_data[i,:,:,:,:1]
    x = np.repeat(x, time_steps-1,axis=3)
    y = in_data[i,:,:,:,1:time_steps]

    x = torch.FloatTensor(x)
    y = torch.FloatTensor(y)

    return x, y

def model_result(model, data, model_print=True):
    x_out = []
    y_out = []
    model_out = []
    print(data.shape)
    print(data[0].shape)
    for i in range(data.shape[0]):
        x, y = get_xy(data, i)
        xin = x.unsqueeze(0).to(device)
        out = model(xin).detach().cpu().numpy()
        x_out.append( x[:,:,:,0].cpu().numpy() )
        y_out.append(y[:,:,:,:].cpu().numpy())
        # y_out.append(_y_out[..., np.newaxis])
        model_out.append( out[0] )

    x_out = np.stack(x_out)
    y_out = np.stack(y_out)
    model_out = np.stack(model_out)

    if model_print:
        print('x_out shape:', x_out.shape)
        print('y_out shape:', y_out.shape)
        print('model_out shape:', model_out.shape)

    return {'x':x_out, 'y':y_out, 'model':model_out}

pred = model_result(model, data_etau_truth, model_print=True)


In [ ]:
# inverse the etau transformation on the predicted data
dat_modeled = pred['model']
tau0 = 0.5
tau_step = 0.1
tau_scaling = tau0 + np.arange(dat_modeled.shape[-1]) * tau_step
dat_modeled[:,0,:,:,:] /= tau_scaling

dat_truth = pred['y']
dat_truth[:,0,:,:,:] /= tau_scaling

print(dat_modeled.shape, dat_truth.shape)

In [ ]:
# graph the distributions of the energy and the shapes and values of the envelopes
# optionally save the results in the output directory as PDF files
for i_event in range(2):
    itlast = 60
    plot_three_bins_contour(dat_modeled, dat_truth, iT=(0,itlast-27,itlast-3), event=i_event, save=f'{odir}/FNOall_cen_9p6_{i_event}.pdf')